In [67]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [68]:
import os 
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from statsmodels.stats.diagnostic import het_arch

In [69]:
from proj.features import transforms
from proj.data.storage import LocalStorage
from proj.evaluation.backtester import Backtester
from proj.evaluation.engine import greedy_forward_feature_selection, greedy_hyperparam_tuning, model_selection, evaluate_models
from proj.utils.paths import find_project_root, build_paths


In [70]:
repo_root = find_project_root()
paths = build_paths(repo_root)

In [71]:
style_path = os.path.join(paths['SRC'], 'proj', 'utils', 'styler.mplstyle')
plt.style.use(style_path)

In [72]:
storage = LocalStorage(base_dir=Path("../data"))


In [73]:
df = storage.read_parquet('silver/equities_daily.parquet')

In [74]:
df

ticker,XLE,SPY,HYG_ret,HYG_absret,TLT_ret,TLT_absret,UNG_ret,UNG_absret,USO_ret,USO_absret
timestamp,,,,,,,,,,
2021-01-05 00:00:00+00:00,16.415472,346.627777,0.034432,0.034432,-0.745494,0.745494,3.432474,3.432474,4.705324,4.705324
2021-01-06 00:00:00+00:00,16.916298,348.700104,-0.183792,0.183792,-2.074169,2.074169,0.509947,0.509947,0.499639,0.499639
2021-01-07 00:00:00+00:00,17.164640,353.880859,0.287016,0.287016,-0.885357,0.885357,-1.331306,1.331306,1.223786,1.223786
2021-01-08 00:00:00+00:00,17.143944,355.897186,0.148910,0.148910,-0.323273,0.323273,0.514142,0.514142,2.573045,2.573045
2021-01-11 00:00:00+00:00,17.412981,353.498199,-0.539389,0.539389,-0.165337,0.165337,3.526818,3.526818,-0.480980,0.480980
...,...,...,...,...,...,...,...,...,...,...
2026-01-05 00:00:00+00:00,46.889999,687.719971,0.259980,0.259980,0.492866,0.492866,-3.630625,3.630625,1.810658,1.810658
2026-01-06 00:00:00+00:00,45.639999,691.809998,0.024730,0.024730,-0.206021,0.206021,-3.055675,3.055675,-2.465344,2.465344
2026-01-07 00:00:00+00:00,45.130001,689.580017,-0.024730,0.024730,0.582628,0.582628,4.337193,4.337193,-1.056505,1.056505


In [75]:
# inclusive: keep rows on/after Jan 8
out = df.loc[df.index >= "2025-01-08", ["XLE"]].dropna()

# exclusive: strictly after Jan 8
out = df.loc[df.index > "2025-01-08", ["XLE"]].dropna()


In [ ]:
df

,run_id,asof_date_et,forecast_date_et,asof_close_utc,model,predicted_value
ts,,,,,,
2026-01-09 21:00:00+00:00,2026-01-09T011135Z,2026-01-08,2026-01-09,2026-01-08 21:00:00+00:00,EWMA_094,0.915678
2026-01-09 21:00:00+00:00,2026-01-09T011135Z,2026-01-08,2026-01-09,2026-01-08 21:00:00+00:00,GARCH_11,1.591384
2026-01-09 21:00:00+00:00,2026-01-09T011135Z,2026-01-08,2026-01-09,2026-01-08 21:00:00+00:00,GARCH_X,1.534353
2026-01-09 21:00:00+00:00,2026-01-09T011135Z,2026-01-08,2026-01-09,2026-01-08 21:00:00+00:00,HAR_RV,1.193189
2026-01-09 21:00:00+00:00,2026-01-09T011135Z,2026-01-08,2026-01-09,2026-01-08 21:00:00+00:00,HAR_RV_X,1.338802
2026-01-12 21:00:00+00:00,2026-01-11T230212Z,2026-01-09,2026-01-12,2026-01-09 21:00:00+00:00,EWMA_094,0.931431
2026-01-12 21:00:00+00:00,2026-01-11T230212Z,2026-01-09,2026-01-12,2026-01-09 21:00:00+00:00,GARCH_11,1.524962
2026-01-12 21:00:00+00:00,2026-01-11T230212Z,2026-01-09,2026-01-12,2026-01-09 21:00:00+00:00,GARCH_X,1.040604
2026-01-12 21:00:00+00:00,2026-01-11T230212Z,2026-01-09,2026-01-12,2026-01-09 21:00:00+00:00,HAR_RV,1.138442


In [ ]:
wide = df.pivot(
    columns="model",
    values="predicted_value"
)

In [ ]:
wide

model,EWMA_094,GARCH_11,GARCH_X,HAR_RV,HAR_RV_X
ts,,,,,
2026-01-09 21:00:00+00:00,0.915678,1.591384,1.534353,1.193189,1.338802
2026-01-12 21:00:00+00:00,0.931431,1.524962,1.040604,1.138442,0.910833


In [ ]:
gold = storage.read_parquet('external/modeling_table.parquet')

In [ ]:
gold[['rv_xle']].merge(wide, left_index=True, right_index=True)

,rv_xle,EWMA_094,GARCH_11,GARCH_X,HAR_RV,HAR_RV_X
2026-01-09 21:00:00+00:00,0.917744,0.915678,1.591384,1.534353,1.193189,1.338802


In [ ]:
# target_ann_vol = 0.15
# target_daily_vol = target_ann_vol / np.sqrt(252)
# w_max = 1.0          # no leverage; set >1 if you allow it
# w_min = 0.0

# w = (target_daily_vol / f_vol).clip(lower=w_min, upper=w_max)

# # strategy returns
# strat_ret = w * ret
# equity = (1 + strat_ret.fillna(0)).cumprod()